In [ ]:
!pip install transformers datasets accelerate scikit-learn -q

In [16]:
import re
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("Abdelrahman-Rezk/Arabic_Dialect_Identification")

def clean_tweet(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
train_df['text_clean'] = train_df['text'].apply(clean_tweet)
val_df['text_clean'] = val_df['text'].apply(clean_tweet)
test_df['text_clean'] = test_df['text'].apply(clean_tweet)

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")

train_encodings = tokenizer(list(train_df['text_clean']), truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(list(val_df['text_clean']), truncation=True, padding=True, max_length=64)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/720k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
import torch

class DialectDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DialectDataset(train_encodings, list(train_df['label']))
val_dataset = DialectDataset(val_encodings, list(val_df['label']))

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2", num_labels=18
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"macro_f1": f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,1.460032,1.423906,0.512685
2,1.310051,1.335067,0.555821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=27504, training_loss=1.5165636150243325, metrics={'train_runtime': 3383.0014, 'train_samples_per_second': 260.155, 'train_steps_per_second': 8.13, 'total_flos': 2.8949794792372224e+16, 'train_loss': 1.5165636150243325, 'epoch': 2.0})

In [ ]:
# second training

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2", num_labels=18
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,1.456383,1.417091,0.514917
2,1.303177,1.308467,0.563834
3,1.142090,1.274088,0.579536


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Macro F1
1,1.456383,1.417091,0.514917
2,1.303177,1.308467,0.563834
3,1.142090,1.274088,0.579536
4,1.021727,1.270098,0.585247


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=55008, training_loss=1.3095720426470405, metrics={'train_runtime': 6892.2695, 'train_samples_per_second': 255.389, 'train_steps_per_second': 7.981, 'total_flos': 5.789958958474445e+16, 'train_loss': 1.3095720426470405, 'epoch': 4.0})

In [17]:
!pip install arabert farasapy -q

from arabert.preprocess import ArabertPreprocessor

model_name = "aubmindlab/bert-base-arabertv02-twitter"
arabert_prep = ArabertPreprocessor(model_name=model_name)

for d in (train_df, val_df, test_df):
    d['text_clean'] = d['text'].apply(arabert_prep.preprocess)

In [18]:
for i in range(3):
    print(train_df['text'].iloc[i])
    print(train_df['text_clean'].iloc[i])
    print()

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    has_at = d['text'].str.contains('@', regex=False)
    has_user = d['text_clean'].str.contains('مستخدم', regex=False)
    empty = (d['text_clean'].str.strip() == '').sum()
    print(name, "aligned:", round((has_at == has_user).mean(), 3), "| empty:", empty)

@Ahmed_Hamza27 اتمنى صراحة و خصوصا #طال_عمره بينا وبينه قصة عشق قديمة😍😍
[مستخدم] اتمنى صراحة و خصوصا # طال _ عمره بينا وبينه قصة عشق قديمة 😍 😍

@rJiM4CnIFTgml9g @zamnissi طبعا عرفته من اللي بشعار الليڤر
[مستخدم] [مستخدم] طبعا عرفته من اللي بشعار اللي ر

@OmanisFollowers @ooredoo @TRA_OMAN @motc_om هيه هيه م يتغير شي.. كلهم ع بعضهم متولفين😪.. بيجيك مدير فرع قريب منك عشان يتساعد معك عشان بس يثبت جدارته لوظيفيه فكرسيه ويحاول يتعاون معك لكن فالنهايه بترجع ع نفس المشكله والدائرة تدور لا بيقدموا ولا بيأخروا😪😪.. وحدي غاسله ايدي منهم
[مستخدم] [مستخدم] [مستخدم] [مستخدم] هيه هيه م يتغير شي . . كلهم ع بعضهم متولفين 😪 . . بيجيك مدير فرع قريب منك عشان يتساعد معك عشان بس يثبت جدارته لوظيفيه فكرسيه ويحاول يتعاون معك لكن فالنهايه بترجع ع نفس المشكله والدائرة تدور لا بيقدموا ولا بيأخروا 😪 😪 . . وحدي غاسله ايدي منهم

train aligned: 0.997 | empty: 31
val aligned: 0.998 | empty: 2
test aligned: 0.997 | empty: 1


In [20]:
import torch
from sklearn.metrics import f1_score

class DialectDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"macro_f1": f1_score(labels, preds, average='macro')}

In [21]:
train_dataset = DialectDataset(train_encodings, list(train_df['label']))
val_dataset = DialectDataset(val_encodings, list(val_df['label']))
test_dataset = DialectDataset(test_encodings, list(test_df['label']))

print(len(train_dataset), len(val_dataset), len(test_dataset))

440021 9164 8981


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=18)

training_args = TrainingArguments(
    output_dir="./results_twitter",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,1.228484,1.201596,0.593940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
!ls results_twitter

ls: cannot access 'results_twitter': No such file or directory
